In [1]:
from pathlib import Path
import csv
import json

product = {
    "product_name": "insurance_policy_product",
    "domain": "insurance",
    "owner": "insurance_data_owner",
    "region_policy": "IN_ONLY_FOR_RAW_PII",
    "monthly_budget_usd": 500,
    "columns": [
        {"name": "policy_id", "classification": "identifier"},
        {"name": "customer_name", "classification": "pii"},
        {"name": "email", "classification": "pii"},
        {"name": "region", "classification": "residency"},
        {"name": "premium_amount", "classification": "financial"},
        {"name": "policy_status", "classification": "operational"},
    ],
}

requests = [
    {
        "request_id": "REQ-001",
        "consumer": "claims_ops",
        "consumer_type": "human",
        "purpose": "claims investigation",
        "columns": ["policy_id", "customer_name", "policy_status"],
        "region": "IN",
    },
    {
        "request_id": "REQ-002",
        "consumer": "ai_claims_assistant",
        "consumer_type": "agent",
        "purpose": "claim summarization",
        "columns": ["policy_id", "customer_name", "email", "policy_status"],
        "region": "IN",
    },
    {
        "request_id": "REQ-003",
        "consumer": "finance_reporting",
        "consumer_type": "human",
        "purpose": "monthly profitability",
        "columns": ["policy_id", "premium_amount", "region"],
        "region": "US",
    },
    {
        "request_id": "REQ-004",
        "consumer": "global_analytics",
        "consumer_type": "human",
        "purpose": "cross-border customer analysis",
        "columns": ["policy_id", "customer_name", "email"],
        "region": "US",
    },
]

cost_records = [
    {"product": "insurance_policy_product", "workload": "quality_checks", "cost_usd": 72, "business_value": "release confidence"},
    {"product": "insurance_policy_product", "workload": "ai_context_queries", "cost_usd": 184, "business_value": "agent support"},
    {"product": "insurance_policy_product", "workload": "dashboard_refresh", "cost_usd": 131, "business_value": "ops reporting"},
    {"product": "insurance_policy_product", "workload": "ad_hoc_queries", "cost_usd": 176, "business_value": "unplanned analysis"},
]

classification = {col["name"]: col["classification"] for col in product["columns"]}

def decide(req):
    requested_classes = {classification.get(col, "unknown") for col in req["columns"]}
    if "unknown" in requested_classes:
        return "block", "unknown column requested"
    if req["consumer_type"] == "agent" and "pii" in requested_classes:
        return "approval_required", "agent requested raw PII"
    if req["region"] != "IN" and "pii" in requested_classes:
        return "block", "raw PII cannot leave IN region"
    if req["purpose"] == "monthly profitability" and "financial" in requested_classes:
        return "allow", "finance purpose matches financial data"
    return "allow", "metadata policy passed"

decisions = []
for req in requests:
    decision, reason = decide(req)
    decisions.append({**req, "decision": decision, "reason": reason})

total_cost = sum(row["cost_usd"] for row in cost_records)
budget_decision = "warn" if total_cost > product["monthly_budget_usd"] else "allow"

summary = {
    "product": product,
    "decisions": decisions,
    "total_cost_usd": total_cost,
    "monthly_budget_usd": product["monthly_budget_usd"],
    "budget_decision": budget_decision,
}

Path("day-08-policy-decisions.json").write_text(json.dumps(summary, indent=2))

with Path("day-08-finops-cost-model.csv").open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["product", "workload", "cost_usd", "business_value"])
    writer.writeheader()
    writer.writerows(cost_records)

print(json.dumps(summary, indent=2))
print("Saved day-08-policy-decisions.json")
print("Saved day-08-finops-cost-model.csv")

{
  "product": {
    "product_name": "insurance_policy_product",
    "domain": "insurance",
    "owner": "insurance_data_owner",
    "region_policy": "IN_ONLY_FOR_RAW_PII",
    "monthly_budget_usd": 500,
    "columns": [
      {
        "name": "policy_id",
        "classification": "identifier"
      },
      {
        "name": "customer_name",
        "classification": "pii"
      },
      {
        "name": "email",
        "classification": "pii"
      },
      {
        "name": "region",
        "classification": "residency"
      },
      {
        "name": "premium_amount",
        "classification": "financial"
      },
      {
        "name": "policy_status",
        "classification": "operational"
      }
    ]
  },
  "decisions": [
    {
      "request_id": "REQ-001",
      "consumer": "claims_ops",
      "consumer_type": "human",
      "purpose": "claims investigation",
      "columns": [
        "policy_id",
        "customer_name",
        "policy_status"
      ],
      "regi